# ALM-Lite — CNN training (SED + Emotion)

This notebook:
1. Downloads **ESC-50** (AudioSet-style environmental subset) and **RAVDESS Speech**.
2. Trains **two separate** mel-spectrogram CNNs with full validation metrics.
3. **Merges** checkpoints into one file for deployment: `outputs/alm_cnn_merged.pt`.

**Outputs:**
- `outputs/sed_cnn.pt` and `outputs/sed_metrics.json`
- `outputs/emotion_cnn.pt` and `outputs/emotion_metrics.json`
- `outputs/alm_cnn_merged.pt` and `outputs/alm_cnn_merged_metrics.json`

Set the working directory to the **project root** (folder containing `config.yaml`).

In [1]:
import os
import sys
from pathlib import Path

# Project root: folder with config.yaml
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "config.yaml").exists():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("Set ROOT manually to your project folder (contains config.yaml).")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: C:\Users\ASIF\Desktop\chethu all\project jn


## 1. Hyperparameters

Increase `EPOCHS_*` for better accuracy (longer GPU/CPU time).

In [2]:
# Stronger defaults for better validation accuracy (adjust to your GPU/CPU budget)
EPOCHS_SED = 40   # ESC-50 / SED CNN
EPOCHS_EMO = 60   # RAVDESS / emotion CNN
BATCH_SED = 16
BATCH_EMO = 32
LR_SED = 5e-4
LR_EMO = 3e-4
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.05
SEED = 42
N_MELS = 64
TIME_FRAMES = 128

## 2. Download both datasets

- **ESC-50**: Hugging Face cache (no torchcodec; uses bytes in dataset).
- **RAVDESS**: ~1.2 GB from Zenodo into `data/datasets/ravdess/`.

In [3]:
from training.download_datasets import download_esc50_subset_hf, download_ravdess_zip

data_root = ROOT / "data" / "datasets"
data_root.mkdir(parents=True, exist_ok=True)

download_esc50_subset_hf(data_root)
try:
    download_ravdess_zip(data_root)
except Exception as e:
    print("RAVDESS download failed:", e)
    print("Manually download Audio_Speech_Actors_01-24.zip from Zenodo and extract under", data_root / "ravdess")

Repo card metadata block was not found. Setting CardData to empty.


ESC-50 ready (HF cache). Manifest: C:\Users\ASIF\Desktop\chethu all\project jn\data\datasets\esc50_manifest.txt


ravdess.zip: 100%|██████████| 208M/208M [00:44<00:00, 4.71MB/s] 


Extracting...
RAVDESS extracted to: C:\Users\ASIF\Desktop\chethu all\project jn\data\datasets\ravdess


## 3. Train SED CNN (ESC-50)

Metrics include **accuracy**, **balanced accuracy**, **macro/micro/weighted F1**, **per-class precision/recall/F1** (sklearn), and **confusion matrix**.

In [4]:
from training.train_cnn_pipeline import train_esc50_sed

sed_result = train_esc50_sed(
    epochs=EPOCHS_SED,
    batch_size=BATCH_SED,
    lr=LR_SED,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    use_class_weights=True,
    early_stopping_patience=8,
    seed=SEED,
    n_mels=N_MELS,
    time_frames=TIME_FRAMES,
    output_pt=ROOT / "outputs" / "sed_cnn.pt",
    output_metrics_json=ROOT / "outputs" / "sed_metrics.json",
)

Loading ESC-50 (AudioSet-style subset) from Hugging Face...


Repo card metadata block was not found. Setting CardData to empty.


epoch 1/20  train_loss=3.7210  val_acc=0.1150  macro_f1=0.0726
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\sed_cnn.pt
epoch 2/20  train_loss=3.2281  val_acc=0.2000  macro_f1=0.1372
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\sed_cnn.pt
epoch 3/20  train_loss=3.0036  val_acc=0.2575  macro_f1=0.2296
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\sed_cnn.pt
epoch 4/20  train_loss=2.8598  val_acc=0.2525  macro_f1=0.2106
epoch 5/20  train_loss=2.7429  val_acc=0.2850  macro_f1=0.2466
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\sed_cnn.pt
epoch 6/20  train_loss=2.6619  val_acc=0.2675  macro_f1=0.2348
epoch 7/20  train_loss=2.6505  val_acc=0.3450  macro_f1=0.3102
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\sed_cnn.pt
epoch 8/20  train_loss=2.5638  val_acc=0.2775  macro_f1=0.2571
epoch 9/20  train_loss=2.4874  val_acc=0.3275  macro_f1=0.2938
epoch 10/20  train_loss=2.4137  val_ac

In [5]:
import json
import pandas as pd
from IPython.display import display

with open(ROOT / "outputs" / "sed_metrics.json", encoding="utf-8") as f:
    sed_json = json.load(f)

summ = sed_json["summary"]
print("SED — best val accuracy:", summ["best_val_accuracy"])
print("SED — num classes:", summ["num_classes"])

fm = sed_json["final_validation_metrics"]["classification_report_dict"]
rows = []
for k, v in fm.items():
    if k in ("accuracy", "macro avg", "weighted avg") or isinstance(v, dict):
        if isinstance(v, dict) and "precision" in v:
            rows.append({"class": k, **{x: v[x] for x in ("precision", "recall", "f1-score", "support") if x in v}})
pd.set_option("display.max_rows", 60)
display(pd.DataFrame(rows))

print("\nConfusion matrix (rows=true, cols=pred):")
for row in sed_json["final_validation_metrics"]["confusion_matrix"]:
    print(row)

SED — best val accuracy: 0.4275
SED — num classes: 50


,class,precision,recall,f1-score,support
0,airplane,0.500000,0.6250,0.555556,8.0
1,breathing,0.200000,0.1250,0.153846,8.0
2,brushing_teeth,0.538462,0.8750,0.666667,8.0
3,can_opening,0.555556,0.6250,0.588235,8.0
4,car_horn,0.625000,0.6250,0.625000,8.0
5,cat,1.000000,0.1250,0.222222,8.0
6,chainsaw,0.714286,0.6250,0.666667,8.0
7,chirping_birds,0.625000,0.6250,0.625000,8.0
8,church_bells,0.500000,0.7500,0.600000,8.0
9,clapping,0.714286,0.6250,0.666667,8.0



Confusion matrix (rows=true, cols=pred):
[5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## 4. Train Emotion CNN (RAVDESS)

Eight emotion classes from speech filenames (`03-…`).

In [6]:
from training.train_cnn_pipeline import train_ravdess_emotion

emo_result = train_ravdess_emotion(
    data_root=ROOT / "data" / "datasets" / "ravdess",
    epochs=EPOCHS_EMO,
    batch_size=BATCH_EMO,
    lr=LR_EMO,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    use_class_weights=True,
    early_stopping_patience=10,
    seed=SEED,
    n_mels=N_MELS,
    time_frames=TIME_FRAMES,
    output_pt=ROOT / "outputs" / "emotion_cnn.pt",
    output_metrics_json=ROOT / "outputs" / "emotion_metrics.json",
)

epoch 1/25  train_loss=2.0394  val_acc=0.3194  macro_f1=0.2329
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.pt
epoch 2/25  train_loss=1.8581  val_acc=0.3194  macro_f1=0.2437
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.pt
epoch 3/25  train_loss=1.7353  val_acc=0.3299  macro_f1=0.2776
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.pt
epoch 4/25  train_loss=1.6451  val_acc=0.4062  macro_f1=0.3339
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.pt
epoch 5/25  train_loss=1.6058  val_acc=0.2743  macro_f1=0.2071
epoch 6/25  train_loss=1.5122  val_acc=0.4583  macro_f1=0.3724
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.pt
epoch 7/25  train_loss=1.4656  val_acc=0.4549  macro_f1=0.4269
epoch 8/25  train_loss=1.4541  val_acc=0.5035  macro_f1=0.4413
  saved best to C:\Users\ASIF\Desktop\chethu all\project jn\outputs\emotion_cnn.p

In [7]:
from IPython.display import display

with open(ROOT / "outputs" / "emotion_metrics.json", encoding="utf-8") as f:
    emo_json = json.load(f)

summ = emo_json["summary"]
print("Emotion — best val accuracy:", summ["best_val_accuracy"])

fm = emo_json["final_validation_metrics"]["classification_report_dict"]
rows = []
for k, v in fm.items():
    if isinstance(v, dict) and "precision" in v:
        rows.append({"class": k, **{x: v[x] for x in ("precision", "recall", "f1-score", "support") if x in v}})
display(pd.DataFrame(rows))

print("\nConfusion matrix:")
for row in emo_json["final_validation_metrics"]["confusion_matrix"]:
    print(row)

Emotion — best val accuracy: 0.5972222222222222


,class,precision,recall,f1-score,support
0,neutral,0.590909,0.684211,0.634146,19.0
1,calm,0.618182,0.894737,0.731183,38.0
2,happy,0.406250,0.342105,0.371429,38.0
3,sad,0.500000,0.421053,0.457143,38.0
4,angry,0.733333,0.564103,0.637681,39.0
5,fearful,0.571429,0.512821,0.540541,39.0
6,disgust,0.609756,0.657895,0.632911,38.0
7,surprised,0.707317,0.743590,0.725000,39.0
8,macro avg,0.592147,0.602564,0.591254,288.0
9,weighted avg,0.593047,0.597222,0.588874,288.0



Confusion matrix:
[13, 3, 1, 2, 0, 0, 0, 0]
[3, 34, 1, 0, 0, 0, 0, 0]
[2, 2, 13, 4, 6, 6, 3, 2]
[4, 12, 0, 16, 0, 5, 0, 1]
[0, 1, 5, 0, 22, 1, 8, 2]
[0, 0, 4, 8, 1, 20, 2, 4]
[0, 3, 4, 2, 1, 0, 25, 3]
[0, 0, 4, 0, 0, 3, 3, 29]


## 5. Merge separate models into one bundle

- **PyTorch**: `outputs/alm_cnn_merged.pt` — keys `sed` and `emotion`.  
- **JSON**: `outputs/alm_cnn_merged_metrics.json` — copies both metric reports for your thesis/report.

In [8]:
from training.merge_cnn_models import merge_cnn_checkpoints

merge_cnn_checkpoints(
    ROOT / "outputs" / "sed_cnn.pt",
    ROOT / "outputs" / "emotion_cnn.pt",
    ROOT / "outputs" / "alm_cnn_merged.pt",
    sed_metrics_json=ROOT / "outputs" / "sed_metrics.json",
    emotion_metrics_json=ROOT / "outputs" / "emotion_metrics.json",
    merged_metrics_json=ROOT / "outputs" / "alm_cnn_merged_metrics.json",
)
print("Merged:", ROOT / "outputs" / "alm_cnn_merged.pt")
print("Merged metrics JSON:", ROOT / "outputs" / "alm_cnn_merged_metrics.json")

Merged: C:\Users\ASIF\Desktop\chethu all\project jn\outputs\alm_cnn_merged.pt
Merged metrics JSON: C:\Users\ASIF\Desktop\chethu all\project jn\outputs\alm_cnn_merged_metrics.json


C:\Users\ASIF\Desktop\chethu all\project jn\training\merge_cnn_models.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sed = torch.load(sed_path, map_location="cpu")
C:\

In [9]:
# Optional: load merged checkpoint in Python
import torch
m = torch.load(ROOT / "outputs" / "alm_cnn_merged.pt", map_location="cpu")
print("Keys:", m.keys())
print("SED kind:", m["sed"].get("kind"))
print("Emotion kind:", m["emotion"].get("kind"))
print("Meta has sed_metrics:", "sed_metrics" in m.get("meta", {}))

Keys: dict_keys(['format', 'created_utc', 'sed', 'emotion', 'meta'])
SED kind: sed_esc50
Emotion kind: emotion_ravdess
Meta has sed_metrics: True


C:\Users\ASIF\AppData\Local\Temp\ipykernel_1824\172038030.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m = torch.load(ROOT / "outputs" / "alm_cnn_merged.pt", map_loca